Exercise to understand K-means clustering

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.cluster import KMeans
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
%matplotlib inline
import datetime as dt
from matplotlib.pyplot import figure
plt.rcParams['figure.figsize'] = (14, 8)



In [ ]:
# Load the dataset.  
df = pd.read_csv("data/Mall_Customers.csv")
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
# Good practice is also to rename columns so its easy to reference

df.rename(columns={'CustomerID': 'customer', 'Gender': 'gender', 'Age': 'age', 'Annual Income (k$)': 'income', 'Spending Score (1-100)': 'spend'}, inplace=True)

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
## Histogram of Age
plt.rcParams['figure.figsize'] = (11, 8)
sns.histplot(df.age)
plt.xlabel('Age')
plt.ylabel('Count')

In [ ]:
## Histogram of Income
plt.rcParams['figure.figsize'] = (11, 8)
sns.histplot(df.income)
plt.xlabel('Income')
plt.ylabel('Count')
plt.show()

In [ ]:
## Histogram of spending score

sns.histplot(df.spend)
plt.xlabel('Spending Score')
plt.ylabel('Count')
plt.show()

In [ ]:
#Labels for Gender
gender = df["gender"].unique().tolist()
gender

In [ ]:
# Count of each gender
gender_count = df.gender.value_counts()
gender_count

In [ ]:
# Pie Plot of the Gender
plt.pie(gender_count, labels = gender, autopct = '%.1f%%')
plt.show()

In [ ]:
# Pairplot
sns.pairplot(df)
plt.show()


In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)
correlation_matrix = df_encoded.corr()
print(correlation_matrix)

In [ ]:
# Correlation HeatMap
plt.rcParams['figure.figsize'] = (8, 6)
sns.heatmap(df_encoded.corr(), annot=True)
plt.show()

In [ ]:
# Lets try to segment by age, income & spending score

# Creating the feature vector

features = df.drop(["customer", "gender"], axis =1)
features.shape

In [ ]:
features.head()

In [ ]:
# Elbow Analysis to find the approximate number of clusters
# (WCSS) Within-Cluster-Sum-of-Squares - for each loop of cluster (1-14)  WCSS is calculated 

kmeans = KMeans()
wcss = []
K = range(1, 15)

for k in K:
    kmeans = KMeans(n_clusters=k).fit(features)
    wcss.append(kmeans.inertia_)
    
plt.plot(K, wcss, "bo-")
plt.xlabel("K Values")
plt.ylabel("WCSS")
plt.show()


In [ ]:
# Lets go with k = 4 

k4 = KMeans(n_clusters = 4, init = 'k-means++', max_iter = 300, n_init = 10, random_state = 0)
k4_pred = k4.fit_predict(features)
k4_pred

In [ ]:
# Add the predictions to the data frame back with featrures

features["cluster"] = k4_pred
features.head()

In [ ]:
# Boxplot of Age by clusters
# plt.rcParams['figure.figsize'] = (12, 6)
sns.boxplot(data=features, x='cluster', y = 'age')
plt.xlabel('clusters')
plt.ylabel('age')
plt.show()

In [ ]:
# Boxplot of Income by clusters
# plt.rcParams['figure.figsize'] = (12, 6)
sns.boxplot(data=features, x='cluster', y = 'income')
plt.xlabel('clusters')
plt.ylabel('income')
plt.show()

In [ ]:
# Boxplot of Spend by clusters
plt.rcParams['figure.figsize'] = (12, 6)
sns.boxplot(data=features, x='cluster', y = 'spend')
plt.xlabel('cluster')
plt.ylabel('spend')
plt.show()

In [ ]:
# scatterplot Income vs Spend based on cluster

plt.figure(figsize=(8,8))
sns.scatterplot(data = features, x='income', y='spend', hue='cluster', palette="deep")
plt.show()

In [ ]:
# Scatterplot Age vs Spend based on cluster

plt.figure(figsize=(8,8))
sns.scatterplot(data = features, x='age', y='spend', hue='cluster', palette="deep")
plt.show()

In [ ]:
# Groupby Clusters and average

segments = features.groupby(by="cluster").mean()
segments

##### Cluster 0 and 1 are low to medium spenders in the age group 40-45
##### Cluster 2 - high spend in the 30 age group
##### Cluster 3 - high spend in the mid 20 age group

####  Another Example

In [ ]:


df = pd.read_csv("data/retail_data.csv", encoding='ISO-8859-1')
df.head()

In [ ]:
df.shape

In [ ]:
df.tail()

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
#df['Customer ID'] = df['Customer ID'].astype(str)
#df.dtypes

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df.InvoiceDate.str.upper(), format='%m/%d/%Y %H:%M', yearfirst=False)
#df["InvoiceDate"] = df["InvoiceDate"].dt.date

In [ ]:
df.head()

In [ ]:
# drop null values

df.dropna(inplace=True)

In [ ]:
df.shape

In [ ]:
# Lets just just check the orders placed by 1 random customer
df.loc[df['Customer ID'] == 17530]

In [ ]:
# number of unique customers

df['Customer ID'].nunique()

In [ ]:
# Different countries 

df['Country'].value_counts()

In [ ]:
# check the observations that have negative value for Quantity 

df.loc[df['Quantity'] < 0].head(10)

In [ ]:
# check the Invoices that starts with the letter "C"

df.loc[df['Invoice'].str.startswith("C",na=False), ["Quantity","Price"]].describe()

In [ ]:
# Removing invoices with "C"

df = df[~df["Invoice"].str.contains("C", na=False)]

In [ ]:
# Calculate Revenue

df['revenue'] = df.Quantity*df.Price

In [ ]:
df.head()

In [ ]:
# 2 days will be added to "today's date" to eliminate the local time differences between stores and to make sure
# that Recency is always above 0
import datetime as dt
max_date = df['InvoiceDate'].max() + pd.DateOffset(days=2)
max_date

In [ ]:
# RFM Data setup

# Finds days from recent order

df["days_from_recent_order"] = max_date - df["InvoiceDate"]
df.head()

In [ ]:
# Calculate Recency 

recency = df.groupby('Customer ID', as_index=False)['days_from_recent_order'].min()
recency.head()

In [ ]:
# Extract only Days

recency['days_from_recent_order'] = recency['days_from_recent_order'].dt.days
recency.head()

In [ ]:
# Frequency
frequency = df.groupby('Customer ID', as_index=False)['Invoice'].count()
frequency.head()

In [ ]:
# Monetary
monetary = df.groupby('Customer ID', as_index=False)['revenue'].sum()
monetary.head()

In [ ]:
# Merge all data to create the RFM data table

rfm = pd.merge(recency, frequency, on="Customer ID", how="inner")
rfm = pd.merge(rfm, monetary, on="Customer ID", how="inner")

rfm.head()

In [ ]:
rfm.dtypes

In [ ]:
# Rename columns for clarity
rfm.columns = ['CustomerID', 'recency', 'frequency', 'monetary']
rfm.head()

In [ ]:
rfm.describe()

In [ ]:
# Z-score standardization  - Range is between -3 & +3 

from sklearn.preprocessing import StandardScaler

# create a scaler object
z_score = StandardScaler()

rfm1 = rfm.iloc[:,1:]

# fit, transform and store in 1 shot

rfm_normalized = z_score.fit_transform(rfm1)

rfm_scaled = pd.DataFrame(rfm_normalized )
rfm_scaled.columns = ['recency', 'frequency', 'monetary']
rfm_scaled.describe()

In [ ]:
# Elbow Analysis to find the approximate number of clusters
# (WCSS) Within-Cluster-Sum-of-Squares - for each loop of cluster (1-14)  WCSS is calculated 

kmeans = KMeans()
wcss = []
K = range(1, 15)

for k in K:
    kmeans = KMeans(n_clusters=k).fit(rfm_scaled)
    wcss.append(kmeans.inertia_)
    
plt.plot(K, wcss, "bo-")
plt.xlabel("K Values")
plt.ylabel("WCSS")
plt.show()

In [ ]:
# Lets go with k = 3 

k3 = KMeans(n_clusters = 3, init = 'k-means++', max_iter = 300, n_init = 10, random_state = 0)
k3_pred = k3.fit_predict(rfm_scaled)
k3_pred

In [ ]:
# Add the predictions to the data frame back with featrures

rfm_scaled["cluster"] = k3_pred
rfm_scaled.head()

In [ ]:
# 2d scatterplot

plt.figure(figsize=(8,8))
sns.scatterplot(data = rfm_scaled, x='frequency', y='monetary', hue='cluster', palette="deep")
plt.show()

In [ ]:
# Cluster Centers

centers = k3.cluster_centers_
centers

In [ ]:
# Storing the cluster centers in a dataframe
#cluster_centers = pd.DataFrame(cluster_centers)
cluster_centers = pd.DataFrame(centers)
# naming the columns 
cluster_centers.columns = ['recency', 'frequency', 'monetary']
cluster_centers.head()

In [ ]:
# 2d scatterplot

plt.figure(figsize=(8,8))
sns.scatterplot(data = rfm_scaled, x='recency', y='frequency', hue='cluster', palette="deep")
sns.scatterplot(data = cluster_centers, x='recency', y='frequency', s=200, color="0", marker="o")
plt.show()

In [ ]:
# 2d scatterplot

plt.figure(figsize=(8,8))
sns.scatterplot(data = rfm_scaled, x='recency', y='monetary', hue='cluster', palette="deep")
sns.scatterplot(data = cluster_centers, x='recency', y='monetary', s=200, color="0", marker="o")
plt.show()

In [ ]:
# 2d scatterplot

plt.figure(figsize=(8,8))
sns.scatterplot(data = rfm_scaled, x='frequency', y='monetary', hue='cluster', palette="deep")
sns.scatterplot(data = cluster_centers, x='frequency', y='monetary', s=200, color="0", marker="o")
plt.show()

In [ ]:
# 3D plot
fig = plt.figure(figsize = (8, 8))
ax = plt.axes(projection ="3d")
cmap = mcolors.ListedColormap(["blue", "green", "red"])
ax.scatter3D(cluster_centers.recency, cluster_centers.frequency, cluster_centers.monetary, s=300, color="0", marker="o")
ax.scatter3D(rfm_scaled.recency, rfm_scaled.frequency, rfm_scaled.monetary, c=rfm_scaled.cluster, cmap=cmap)
ax.set_xlabel('Recency')
ax.set_ylabel('Frequency')
ax.set_zlabel('Monetary')
plt.title('RFM in 3D with Clusters', size=15)
ax.set(facecolor='white')
plt.show()

Author: Mani Kannappan